# 02 — Diagnose weaknesses

For each worst-scoring brief, use the LLM to explain **why** the lowest criterion scored low and which template section contributed. Then summarise common failure patterns across the sample.

**Input:** `01-worst-briefs.jsonl` from this improvement run, plus the current `paper_brief_template.md`.

**Output:** `02-diagnosis.jsonl` (per-paper) and `02-diagnosis-summary.md` (cross-paper patterns).

See [paper-brief-improvement.md](../../docs/specs/paper-brief-improvement.md) step 2.

In [ ]:
# Required chat model id.
MODEL = ""

# Improvement run folder under data/paper_brief_improvement/.
# Leave empty to use the latest folder that has 01-worst-briefs.jsonl.
RUN_ID = ""

In [ ]:
from __future__ import annotations

import json
import os
import re
from pathlib import Path

from IPython.display import Markdown, display


def repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "src" / "paper_reviewer").is_dir() and (
            candidate / "pyproject.toml"
        ).is_file():
            return candidate
    raise RuntimeError(
        "Cannot find the repo root. Start Jupyter with `just notebooks` "
        "so the kernel can see /workspace."
    )


REPO_ROOT = repo_root()
IMPROVEMENT_PARENT = REPO_ROOT / "data" / "paper_brief_improvement"

print(f"repo root: {REPO_ROOT}")
print(f"improvement parent: {IMPROVEMENT_PARENT}")

In [ ]:
_RUN_ID_PATTERN = re.compile(r"^\d{8}T\d{6}Z_.+$")
CRITERIA = ("faithfulness", "completeness", "conciseness", "topic_agnostic")


def load_jsonl(path: Path) -> list[dict]:
    rows: list[dict] = []
    with path.open(encoding="utf-8") as fh:
        for line in fh:
            stripped = line.strip()
            if stripped:
                rows.append(json.loads(stripped))
    return rows


def resolve_run_dir(run_id: str) -> Path:
    if run_id:
        d = IMPROVEMENT_PARENT / run_id
        if not (d / "01-worst-briefs.jsonl").is_file():
            raise FileNotFoundError(f"No 01-worst-briefs.jsonl in {d}")
        return d
    candidates = sorted(
        (
            p.parent
            for p in IMPROVEMENT_PARENT.glob("*/01-worst-briefs.jsonl")
            if _RUN_ID_PATTERN.match(p.parent.name)
        ),
        key=lambda d: d.name,
    )
    if not candidates:
        raise FileNotFoundError(
            "No improvement runs with 01-worst-briefs.jsonl found under "
            + str(IMPROVEMENT_PARENT)
        )
    return candidates[-1]


def worst_criteria(evaluation: dict) -> list[str]:
    """Return criterion names that share the lowest score."""
    scores = {}
    for c in CRITERIA:
        entry = evaluation.get(c, {})
        s = entry.get("score")
        if isinstance(s, (int, float)) and not isinstance(s, bool):
            scores[c] = s
    if not scores:
        return list(CRITERIA)
    min_score = min(scores.values())
    return [c for c, s in scores.items() if s == min_score]

In [ ]:
model = MODEL.strip()
assert model, "MODEL must be set to a chat model id"
os.environ["OPENAI_MODEL"] = model

run_dir = resolve_run_dir(RUN_ID)
print(f"run dir: {run_dir.relative_to(REPO_ROOT)}")

worst_rows = load_jsonl(run_dir / "01-worst-briefs.jsonl")
print(f"worst briefs loaded: {len(worst_rows)}")

from paper_reviewer.topic_scope.generate_paper_brief.llm import (
    load_paper_brief_template,
)

template_text = load_paper_brief_template()
print(f"template length: {len(template_text)} chars")

In [ ]:
from paper_reviewer.topic_scope.generate_paper_brief.llm import (
    resolve_openai_base_url,
    resolve_openai_model,
)


def _make_client():
    from openai import OpenAI

    api_key = os.environ.get("OPENAI_API_KEY") or "placeholder"
    base_url = resolve_openai_base_url(
        os.environ.get("OPENAI_BASE_URL"),
        in_container=Path("/.dockerenv").exists(),
    )
    return OpenAI(api_key=api_key, base_url=base_url)


def _chat(client, system: str, user: str) -> str:
    resolved_model = resolve_openai_model(os.environ.get("OPENAI_MODEL"))
    resp = client.chat.completions.create(
        model=resolved_model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=0.3,
    )
    return resp.choices[0].message.content or ""


client = _make_client()
print("LLM client ready")

In [ ]:
DIAGNOSIS_SYSTEM = """You are a scientific-writing quality analyst.

You receive:
1. A paper brief (structured JSON) generated from a scientific article.
2. The full text of that article.
3. A G-Eval evaluation with per-criterion scores and reasoning.
4. The template (system prompt) that was used to generate the brief.

Your task: explain WHY the weakest criterion (or criteria) scored low.
Focus on:
- What the brief got wrong or missed relative to the full text.
- Which section or instruction in the template failed to prevent the error,
  or which missing instruction would have helped.
- Provide a short quote from the brief and/or full text to support your diagnosis.

Be specific and actionable. Do NOT rewrite the brief or the template.
Output plain text (no JSON, no Markdown headers)."""


def build_diagnosis_user_msg(row: dict, full_text: str, template: str) -> str:
    wc = worst_criteria(row["evaluation"])
    parts = [
        f"## Worst criterion/criteria: {', '.join(wc)}",
        "",
        "## Evaluation (scores + reasoning)",
        json.dumps(row["evaluation"], indent=2, ensure_ascii=False),
        "",
        "## Brief",
        json.dumps(row["brief"], indent=2, ensure_ascii=False),
        "",
        "## Template used to generate the brief",
        template,
        "",
        "## Full text of the article (truncated to first 12000 chars)",
        full_text[:12000],
    ]
    return "\n".join(parts)


print("diagnosis prompt ready")

In [ ]:
diagnosis_rows: list[dict] = []

for i, row in enumerate(worst_rows):
    doi = row["doi"]
    corpus_path = REPO_ROOT / row["corpus_file"]
    if not corpus_path.is_file():
        print(f"[{i+1}/{len(worst_rows)}] SKIP {doi}: corpus file missing")
        diagnosis_rows.append({"doi": doi, "worst_criteria": [], "diagnosis": None, "error": "corpus file missing"})
        continue

    full_text = corpus_path.read_text(encoding="utf-8")
    wc = worst_criteria(row["evaluation"])
    user_msg = build_diagnosis_user_msg(row, full_text, template_text)

    print(f"[{i+1}/{len(worst_rows)}] diagnosing {doi} (worst: {', '.join(wc)}) ...", end=" ", flush=True)
    try:
        diagnosis = _chat(client, DIAGNOSIS_SYSTEM, user_msg)
        diagnosis_rows.append({"doi": doi, "worst_criteria": wc, "diagnosis": diagnosis})
        print(f"ok ({len(diagnosis)} chars)")
    except Exception as exc:
        print(f"ERROR: {exc}")
        diagnosis_rows.append({"doi": doi, "worst_criteria": wc, "diagnosis": None, "error": str(exc)})

print(f"\ndiagnosed: {sum(1 for r in diagnosis_rows if r.get('diagnosis'))} / {len(diagnosis_rows)}")

In [ ]:
diag_path = run_dir / "02-diagnosis.jsonl"
with diag_path.open("w", encoding="utf-8") as fh:
    for row in diagnosis_rows:
        fh.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"wrote {len(diagnosis_rows)} rows to {diag_path.relative_to(REPO_ROOT)}")

In [ ]:
SUMMARY_SYSTEM = """You are a scientific-writing quality analyst.

You receive:
1. A set of per-paper diagnoses explaining why specific G-Eval criteria scored low on generated paper briefs.
2. The template (system prompt) used to generate those briefs.

Your task: identify COMMON FAILURE PATTERNS across the sample.
For each pattern:
- Name the pattern concisely.
- List which G-Eval criteria it affects.
- Point to the specific template section (or missing instruction) responsible.
- Count how many of the diagnosed papers exhibit it.

Output Markdown with a heading per pattern. Be specific and actionable.
Do NOT propose fixes (that is step 3). Only diagnose."""

successful_diags = [r for r in diagnosis_rows if r.get("diagnosis")]
if not successful_diags:
    raise RuntimeError("No successful diagnoses to summarise")

diag_block = "\n\n---\n\n".join(
    f"### DOI: {r['doi']}\nWorst criteria: {', '.join(r['worst_criteria'])}\n\n{r['diagnosis']}"
    for r in successful_diags
)

summary_user = (
    f"## Per-paper diagnoses ({len(successful_diags)} papers)\n\n"
    f"{diag_block}\n\n"
    f"## Template used to generate the briefs\n\n{template_text}"
)

print(f"summarising {len(successful_diags)} diagnoses ...", flush=True)
summary_text = _chat(client, SUMMARY_SYSTEM, summary_user)
print(f"summary: {len(summary_text)} chars")

In [ ]:
summary_path = run_dir / "02-diagnosis-summary.md"
summary_path.write_text(summary_text, encoding="utf-8")
print(f"wrote {summary_path.relative_to(REPO_ROOT)}")

display(Markdown(summary_text))